# Fine-tune QLoRA — Qwen2.5-3B (tiếng Việt tài liệu học thuật)
Chạy trên **Google Colab (GPU T4 free)**. Kết quả tải về import vào Ollama.
*Chuẩn bị:* upload file `dataset.jsonl` vào Colab (mỗi dòng 1 mẫu chat):
```json
{"instruction": "Tóm tắt tài liệu sau.", "input": "nội dung...", "output": "tóm tắt..."}
{"instruction": "Kiểm duyệt tài liệu này.", "input": "nội dung...", "output": "{\"verdict\":\"rejected\",\"reason\":\"...\"}"}
```
*Khuyến nghị:* 300-1000 mẫu chất lượng, giữ lại ~10% làm tập test.

In [ ]:
# 1. Cài Unsloth (nhanh + tiết kiệm VRAM)
!pip install -q unsloth
# Khởi động lại runtime nếu được yêu cầu

In [ ]:
# 2. Tải model 4-bit + gắn LoRA
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=1024,          # giảm xuống 512 nếu OOM
    dtype=None, load_in_4bit=True,
    device_map="auto",
)
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

In [ ]:
# 3. Nạp dataset (file dataset.jsonl bạn upload lên)
import json
from datasets import Dataset

samples = []
with open("dataset.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            samples.append(json.loads(line))

# Tách train/test
import random
random.shuffle(samples)
split = int(len(samples) * 0.9)
train_data, test_data = samples[:split], samples[split:]
print(f"Train: {len(train_data)}, Test: {len(test_data)}")

def build_prompt(example):
    if example.get("input"):
        user = f"{example['instruction']}\n{example['input']}"
    else:
        user = example["instruction"]
    return {"prompt": user, "completion": example["output"]}

train_ds = Dataset.from_list([build_prompt(x) for x in train_data])

def format_row(row):
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": row["prompt"]},
         {"role": "assistant", "content": row["completion"]}],
        tokenize=False, add_generation_prompt=False,
    )

train_ds = train_ds.map(lambda r: {"text": format_row(r)})

In [ ]:
# 4. Huấn luyện LoRA (2-3 epoch)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=train_ds,
    dataset_text_field="text", max_seq_length=1024,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=2,
        learning_rate=2e-4, fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(), logging_steps=10,
        optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=42,
        output_dir="outputs",
    ),
)
trainer.train()

In [ ]:
# 5. Merge adapter + export GGUF (Q4_K_M) để dùng trong Ollama
model.save_pretrained_merged("merged_model", tokenizer)
model.save_pretrained_gguf(
    "gguf_output", tokenizer,
    quantization_method="q4_k_m",   # ~2GB
)

# In nội dung Modelfile để import vào Ollama
modelfile = model.populate_modelfile("gguf_output/README.md") if False else None
import glob
gguf_path = glob.glob("gguf_output/*.gguf")[0]
print("GGUF:", gguf_path)

In [ ]:
# 6. Tải về máy (chờ download xong)
from google.colab import files
import glob
gguf_path = glob.glob("gguf_output/*.gguf")[0]
files.download(gguf_path)   # ~2GB, lưu vào thư mục dự án

In [ ]:
# 7. (Tuỳ chọn) Đánh giá nhanh trên tập test trước khi dùng
from transformers import TextStreamer
FastLanguageModel.for_inference(model)
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_data[0]["instruction"]}],
    tokenize=False, add_generation_prompt=True)
inputs = tokenizer([text], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))